In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catlog_name}.{bronze_schema}.races"
silver_table = f"{catlog_name}.{silver_schema}.races"

In [0]:
from pyspark.sql import functions as F

In [0]:
races_df = spark.table(bronze_table)

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitID"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

In [0]:
race_renamed_df = (races_selected_df
        .withColumnsRenamed({
            "circuitID": "circuit_id",
            "raceName": "race_name",
            "date": "race_date"
}))

In [0]:
display(race_renamed_df)

In [0]:
race_distinct_df = race_renamed_df.dropDuplicates(["season","round"])

In [0]:
race_final_df = (race_distinct_df
                 .withColumn("race_name", F.initcap(F.col("race_name")
                                                    )
                             )
                 )

In [0]:
display(race_final_df)

In [0]:
(
    race_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))